# License Plate Recognition

Detects vehicle license plates in a video with a fine-tuned **YOLO** model,
reads the characters with **EasyOCR**, validates them against an Indian plate
pattern (`AA00AAA`), corrects common OCR confusions (e.g. `0`↔`O`), and
stabilises the result across frames with a per-plate majority vote.

**Pipeline:** detect → crop → preprocess → OCR → format-correct → validate → stabilise → annotate.

## 1. Imports

In [ ]:
import re
from collections import defaultdict, deque

import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
import easyocr

## 2. Configuration

In [ ]:
INPUT_VIDEO = "video.mp4"
OUTPUT_VIDEO = "output_video.mp4"
WEIGHTS = "license_plate_best.pt"

CONF_THRESH = 0.3      # Minimum YOLO detection confidence.
USE_GPU = True         # EasyOCR falls back to CPU automatically if no GPU.

# Expected pattern: two letters, two digits, three letters (e.g. "KL07ABC").
PLATE_PATTERN = re.compile(r"^[A-Z]{2}[0-9]{2}[A-Z]{3}$")
OCR_ALLOWLIST = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

# Characters OCR commonly confuses, in each direction.
NUM_TO_ALPHA = {"0": "O", "1": "I", "5": "S", "8": "B"}
ALPHA_TO_NUM = {"O": "0", "I": "1", "S": "5", "B": "8"}

## 3. Load the models\n\nOn the first run, EasyOCR downloads its detection and recognition weights (~100 MB) to a local cache.

In [ ]:
model = YOLO(WEIGHTS)
reader = easyocr.Reader(["en"], gpu=USE_GPU)

## 4. Plate-format correction

Positions 0–1 and 4–6 must be letters; positions 2–3 must be digits.
Characters in the wrong class are remapped with the confusion tables when
possible; anything that cannot be made to fit is discarded.

In [ ]:
def correct_plate_format(ocr_text):
    """Normalise raw OCR text to the expected plate pattern, or return ""."""
    ocr_text = ocr_text.upper().replace(" ", "")
    if len(ocr_text) != 7:
        return ""  # Wrong length: discard.

    corrected = []
    for i, ch in enumerate(ocr_text):
        if i < 2 or i >= 4:  # Letter positions.
            if ch.isalpha():
                corrected.append(ch)
            elif ch.isdigit() and ch in NUM_TO_ALPHA:
                corrected.append(NUM_TO_ALPHA[ch])
            else:
                return ""
        else:  # Digit positions.
            if ch.isdigit():
                corrected.append(ch)
            elif ch.isalpha() and ch in ALPHA_TO_NUM:
                corrected.append(ALPHA_TO_NUM[ch])
            else:
                return ""

    return "".join(corrected)

## 5. OCR on a plate crop

In [ ]:
def preprocess_plate(plate_crop):
    """Convert a plate crop to a high-contrast, upscaled image for OCR."""
    gray = cv2.cvtColor(plate_crop, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return cv2.resize(thresh, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)


def recognize_plate(plate_crop):
    """Read and validate a plate from a cropped image, or return ""."""
    if plate_crop.size == 0:
        return ""

    plate_image = preprocess_plate(plate_crop)
    try:
        ocr_result = reader.readtext(plate_image, detail=0, allowlist=OCR_ALLOWLIST)
    except Exception:
        return ""

    if not ocr_result:
        return ""

    candidate = correct_plate_format(ocr_result[0])
    if candidate and PLATE_PATTERN.match(candidate):
        return candidate
    return ""

## 6. Temporal stabilisation

Per-frame reads are noisy. Each detection box (quantised to tolerate small
jitter) keeps a short history of recent reads, and the most frequent read is
reported.

In [ ]:
class PlateStabilizer:
    """Smooths noisy per-frame reads into a stable plate via majority vote."""

    def __init__(self, history_size=10):
        self._history = defaultdict(lambda: deque(maxlen=history_size))
        self._final = {}

    @staticmethod
    def box_id(x1, y1, x2, y2):
        """Quantise a box so a jittery box maps to one stable key."""
        return f"{x1 // 10}_{y1 // 10}_{x2 // 10}_{y2 // 10}"

    def update(self, box_id, plate):
        """Record a new read and return the current best plate for the box."""
        if plate:
            history = self._history[box_id]
            history.append(plate)
            self._final[box_id] = max(set(history), key=history.count)
        return self._final.get(box_id, "")

## 7. Rendering

In [ ]:
OVERLAY_W, OVERLAY_H = 400, 150


def draw_detection(frame, box, plate_crop, text):
    """Draw the plate box, a zoomed-in overlay, and the read text."""
    x1, y1, x2, y2 = box
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)

    if plate_crop.size == 0:
        return

    zoomed = cv2.resize(plate_crop, (OVERLAY_W, OVERLAY_H))
    oy1 = max(0, y1 - OVERLAY_H - 40)
    ox1 = x1
    oy2, ox2 = oy1 + OVERLAY_H, ox1 + OVERLAY_W

    if oy2 <= frame.shape[0] and ox2 <= frame.shape[1]:
        frame[oy1:oy2, ox1:ox2] = zoomed
        if text:
            cv2.putText(frame, text, (ox1, oy1 - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 0, 0), 6)
            cv2.putText(frame, text, (ox1, oy1 - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 2.0, (255, 255, 255), 3)

## 8. Run on the video

Processes every frame, writes the annotated result to `OUTPUT_VIDEO`, and keeps
the last annotated frame so we can preview it inline afterwards.

In [ ]:
stabilizer = PlateStabilizer()

cap = cv2.VideoCapture(INPUT_VIDEO)
assert cap.isOpened(), f"Could not open input video '{INPUT_VIDEO}'"

fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

frame_count = 0
last_frame = None
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1

    results = model(frame, verbose=False)
    for r in results:
        for box in r.boxes:
            conf = float(box.conf[0])
            if conf < CONF_THRESH:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            plate_crop = frame[y1:y2, x1:x2]

            text = recognize_plate(plate_crop)
            box_id = PlateStabilizer.box_id(x1, y1, x2, y2)
            stable_text = stabilizer.update(box_id, text)

            draw_detection(frame, (x1, y1, x2, y2), plate_crop, stable_text)

    out.write(frame)
    last_frame = frame

cap.release()
out.release()
print(f"Processed {frame_count} frames. Output saved as: {OUTPUT_VIDEO}")

## 9. Preview the last annotated frame

In [ ]:
if last_frame is not None:
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(last_frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()